# 06 · Analyses

1. **Position bias.** Listwise Llama on the same test queries, with candidates in v1's positives-first order, shuffled, and reversed.
2. **Stage-1 diagnostic.** Recall of the true references in Qwen3's top k, for k = 5, 10, 20, 50. This tests v1's "stage-1 recall loss" explanation.
3. **Contamination check.** The LLM's gain over Qwen3 for queries published before vs after Llama 3.2's knowledge cutoff. Meta's model card says: "The pretraining data has a cutoff of December 2023."
4. **LLM reliability.** Listwise outputs that were partial or unparseable.

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")

RESEARCH = Path.cwd().parent if Path.cwd().name == "v2" else Path.cwd()
sys.path.insert(0, str(RESEARCH))

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

from src.figures import save, setup_style
setup_style()

In [ ]:
BENCHES = ["concept", "random"]
SPLIT = "test"
CUTOFF = "2023-12-31"
BASELINE = "qwen"

In [ ]:
import json
from src.pipeline_v2 import load_eval_sets, load_runs
from src.stats_v2 import contamination_split, per_query, position_bias, stage1_recall

C = json.loads((RESEARCH / "v2" / "config.json").read_text())
TAG = f"{C['fields']}_{C['max_chars']}_{C['query_max_chars']}"
runs = {b: load_runs(b, TAG, split=SPLIT) for b in BENCHES}

### 1 · Position bias

If the LLM uses content rather than position, MRR and R-precision should not depend on the input order.

In [ ]:
pb = {}
for bench, df in runs.items():
    pb[bench] = position_bias(df[df["seed"].isin(df[df["order"] != "shuffled"]["seed"].unique())], "llama")
    print(bench); display(pb[bench])

In [ ]:
from src.figures import fig_position_bias
save(fig_position_bias(pb), "06_position_bias")

**Figure (06_position_bias).** Listwise Llama 3.2 3B on the same test queries and seed, with candidates in three input orders. "Positives first" reproduces v1's order. Differences between rows measure position bias.

### 2 · Stage-1 recall (Qwen3)

`queries_full_recall` is the number of queries whose references all survive the cut. Compare it with the two-stage result in `05_stats`.

In [ ]:
recalls = {}
for bench, df in runs.items():
    recalls[bench] = stage1_recall(df, "qwen")
    print(bench); display(recalls[bench])

In [ ]:
from src.figures import fig_stage1_recall
save(fig_stage1_recall(recalls), "06_stage1_recall")

**Figure (06_stage1_recall).** Mean share of each query's cited references that Qwen3 ranks in its top k. Where the curve is close to 100% at the two-stage cut-off, stage 1 loses almost nothing, so a two-stage drop cannot be blamed on stage-1 recall.

### 3 · Contamination check

**Caveat:** almost all queries were published before the cutoff, so the "after" group is tiny. Read this as a limitation, not as evidence either way.

In [ ]:
for bench, df in runs.items():
    ds = load_eval_sets(bench)
    dates = {s["pi_id"]: s["pi_date"] for s in ds["sets"]}
    print(bench); display(contamination_split(per_query(df), dates, CUTOFF, "llama", BASELINE))

### 4 · LLM output reliability

In [ ]:
for bench, df in runs.items():
    llm = df[df["method"].str.startswith(("llama", "qwen→llama")) & (df["method"] != "llama+minilm")]
    print(bench); display(llm.groupby(["method", "order"])["status"].value_counts().unstack(fill_value=0))

### Findings

_Written from the outputs above after the notebook is run._